# VoxCPM Synthetic Data Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfriSpeech/voxcpm-synthetic-data/blob/main/notebooks/voxcpm_synth.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/AfriSpeech/voxcpm-synthetic-data/blob/main/notebooks/voxcpm_synth.ipynb)

Generate synthetic TTS training audio from a text dataset, voice-cloning the
built-in male/female speakers. Output: `audio/*.wav` + `manifest.jsonl` you can
feed to your TTS trainer.

> **Use a GPU runtime.** Colab: `Runtime → Change runtime type → T4 GPU`.
> Kaggle: `Settings → Accelerator → GPU` **and Internet ON**.

## Install

In [ ]:
!apt-get -qq install -y ffmpeg >/dev/null
!git clone -q https://github.com/AfriSpeech/voxcpm-synthetic-data.git
%cd voxcpm-synthetic-data
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — switch to a GPU runtime!'

## (Optional) Hugging Face token
Needed if the model or your dataset is gated/private. Skip otherwise.

In [ ]:
import os, getpass
_tok = getpass.getpass('HF token (optional — Enter to skip): ').strip()
if _tok:
    os.environ['HF_TOKEN'] = _tok
    print('Token set.')
else:
    print('No token set.')

## Configure your run
Point at a text dataset on the Hub and its text column. (List AfriSpeech-org
datasets with `!python -m voxcpm_synth --list-datasets`.)

In [ ]:
DATASET = "ghananlpcommunity/CHANGE-ME"   # <-- your text dataset id
TEXT_COLUMN = "text"                        # <-- the text column
NAME = "demo"                                # output goes to data/demo/

## Preview a few clips
Hear the result before a big run.

In [ ]:
!python -m voxcpm_synth --dataset {DATASET} --text-column {TEXT_COLUMN} --name {NAME} --preview 3

import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'data/{NAME}/preview/*.wav')):
    print(w)
    display(Audio(w))

## Generate
Small demo run (0.2 h). Bump `--hours` for a real run. Re-run the same cell to
**resume** (finished rows are skipped). Add `--voices male|female` or
`--male-pct N` to control speakers, `--instances N` for parallelism.

In [ ]:
!python -m voxcpm_synth --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --hours 0.2 --name {NAME}

## Inspect the output

In [ ]:
import json, glob
from IPython.display import Audio, display

rows = [json.loads(l) for l in open(f'data/{NAME}/manifest.jsonl', encoding='utf-8')]
print(f'{len(rows)} clips, {sum(r["duration"] for r in rows)/3600:.3f} h')
for r in rows[:3]:
    print(f"\n[{r['gender']}] {r['duration']}s  {r['text'][:90]}")
    display(Audio(f"data/{NAME}/{r['file']}"))

## (Optional) Push to your own HF dataset repo
```bash
!python -m voxcpm_synth --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --hours 0.2 --name {NAME} --push your-username/my-synth --token $HF_TOKEN
```
Full options: https://github.com/AfriSpeech/voxcpm-synthetic-data